# Fine-Tuning Mistral 7B with QLoRA

**What:** Fine-tune Mistral 7B on invoice field extraction using QLoRA (4-bit NF4 quantization + LoRA adapters).

**Where:** Kaggle T4 GPU (16GB VRAM).

**Key decisions (from data analysis):**
- `bf16=True` — Mistral 7B natively uses bfloat16; bf16 training avoids GradScaler crash with quantized models
- `max_seq_length=768` — 99%+ of samples are under 700 tokens (from token length analysis)
- `batch_size=2, grad_accum=8` — effective batch of 16, fits T4 VRAM
- `LoRA r=64, alpha=128` — targets all attention + MLP layers (167M trainable / 7.4B total = 2.26%)
- `learning_rate=1e-4` — conservative to prevent training collapse (2e-4 caused loss explosion in earlier runs)
- `save_steps=30` — frequent checkpoints to preserve best model before potential instability

## 1. Install Dependencies

In [ ]:
!pip install -q peft trl bitsandbytes wandb sentencepiece pydantic rapidfuzz

## 2. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3. Clone Code

In [ ]:
!git clone https://github.com/Pushparaj13811/invoice-extraction-mistral-7b-fine-tuning.git

import sys
sys.path.insert(0, '/kaggle/working/invoice-extraction-mistral-7b-fine-tuning')

## 4. Set API Tokens

In [ ]:
import os

os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['WANDB_API_KEY'] = 'YOUR_WANDB_KEY'

In [ ]:
import wandb
wandb.login()
print('W&B logged in')

## 5. Configure Training

Hyperparameters chosen based on data analysis and previous training runs:
- **LoRA rank 64**: 167M trainable parameters across all attention + MLP layers
- **Learning rate 1e-4**: Reduced from 2e-4 which caused loss explosion at step 100-200
- **bf16=True**: Matches Mistral's native dtype, avoids fp16 GradScaler crash with bitsandbytes
- **save_steps=30**: Captures best checkpoint before any potential training instability
- **1 epoch**: Sufficient for ~1,445 samples; 3 epochs caused overfitting and collapse

In [ ]:
from src.training.config import TrainingConfig

config = TrainingConfig(
    model_name='mistralai/Mistral-7B-v0.3',
    lora_r=64,
    lora_alpha=128,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_seq_length=768,
    save_steps=30,
    eval_steps=30,
)

In [ ]:
print(f'Model:          {config.model_name}')
print(f'LoRA:           r={config.lora_r}, alpha={config.lora_alpha}')
print(f'Batch size:     {config.per_device_train_batch_size} x {config.gradient_accumulation_steps} = {config.effective_batch_size}')
print(f'Epochs:         {config.num_train_epochs}')
print(f'Max seq length: {config.max_seq_length}')
print(f'Learning rate:  {config.learning_rate}')
print(f'Save steps:     {config.save_steps}')

## 6. Verify Data

In [ ]:
TRAIN_PATH = '/kaggle/input/datasets/pushparajmehta/llm-pre-train-dataset/train.jsonl'
EVAL_PATH = '/kaggle/input/datasets/pushparajmehta/llm-pre-train-dataset/eval.jsonl'

In [ ]:
from src.data.format import load_jsonl

train_data = load_jsonl(TRAIN_PATH)
eval_data = load_jsonl(EVAL_PATH)
print(f'Train samples: {len(train_data)}')
print(f'Eval samples:  {len(eval_data)}')

## 7. Train

Loads model in 4-bit (QLoRA), applies LoRA adapters, trains with SFTTrainer.
Uses bf16 for stable training — no GradScaler needed.
Checkpoints saved every 30 steps.

In [ ]:
from src.training.train import train

trainer = train(
    config=config,
    train_path=TRAIN_PATH,
    eval_path=EVAL_PATH,
    output_dir='/kaggle/working/outputs/',
)
print('Training complete!')

## 8. Check Checkpoints

Find the best checkpoint by eval loss:

In [ ]:
import json, os

for d in sorted(os.listdir('/kaggle/working/outputs/')):
    state_path = f'/kaggle/working/outputs/{d}/trainer_state.json'
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
        evals = [e for e in state['log_history'] if 'eval_loss' in e]
        if evals:
            best = min(evals, key=lambda x: x['eval_loss'])
            print(f'{d}: eval_loss={best["eval_loss"]:.4f} at step {best["step"]}')

## 9. Save Best Adapter

Copy the final adapter and zip for download:

In [ ]:
import shutil

adapter_path = '/kaggle/working/outputs/final_adapter'
print(f'Adapter files: {os.listdir(adapter_path)}')

shutil.make_archive('/kaggle/working/final_adapter', 'zip', adapter_path)
print('Zipped for download')

## 10. Quick Sanity Check

Test one invoice to verify the model produces valid JSON:

In [ ]:
from src.evaluation.evaluate import load_finetuned_model
import torch

model, tokenizer = load_finetuned_model(config.model_name, adapter_path)
model.eval()

In [ ]:
test_input = 'Invoice #TEST-001\nFrom: Test Corp\nDate: 2024-06-15\nDue: 2024-07-15\nItem: Widget x2 @ $50.00 = $100.00\nTax: $10.00\nTotal: $110.00\nCurrency: USD'

prompt = f'### Instruction:\nExtract all invoice fields from the following invoice text as JSON.\n\n### Input:\n{test_input}\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 11. Download

Download the adapter from Kaggle Output tab, or:

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/final_adapter.zip')